# Sesion 8 — De Datos Crudos a Dataset Analizable
## Diplomado: Machine Learning en Seguros · FC UNAM
### 2 de mayo de 2026  ·  07:00 - 11:00 h  (4 horas)

---

> **Premisa de la sesion:** recibes los archivos crudos de la aseguradora.
> Tu trabajo: convertirlos en un dataset limpio, bien tipado y eficiente,
> resolviendo 7 problemas reales uno por uno.

---

**Prerequisito:** Debe existir la carpeta `datos/` con los archivos CSV.

## Las 7 Dudas que Resolvemos Hoy

| # | Duda | Herramienta |
|---|------|-------------|
| 1 | 46 columnas — ¿cuales necesito? | Taxonomia + `usecols` |
| 2 | Texto sucio: M/F/Masculino/Femenino | `str` operations |
| 3 | Fechas como texto: '15/04/2026' | `pd.to_datetime()` |
| 4 | API de reaseguro devuelve JSON | `read_json()` + `json_normalize()` |
| 5 | 90k filas, 11MB sin esperar | `chunks` + `category` |
| 6 | Downcast rompio precision de primas | Estrategia segura de `dtypes` |
| 7 | ¿Cuando cambiar a Polars? | `polars` — intro y comparativa |

---
## ACT 1 — El Dataset Crudo

### Duda 1: 46 columnas — ¿cuales necesito?

In [7]:
import pandas as pd
import numpy as np
import os, time

# ── Paso 1: Cargar TODO para entender que hay ────────────────────────────────
# Primera regla: antes de descartar, entende lo que tienes
df_todo = pd.read_csv('datos/cartera_polizas.csv', nrows=5)  # solo 5 filas para ver, validar la informacion
print(f'Columnas totales: {len(df_todo.columns)}')
print()
print('Lista de columnas:')
for i, col in enumerate(df_todo.columns, 1):
    print(f'  {i:>2}. {col}')

Columnas totales: 46

Lista de columnas:
   1. id_poliza
   2. num_poliza
   3. id_contrato_interno
   4. folio_emision
   5. id_sistema_legacy
   6. nombre
   7. apellido_paterno
   8. apellido_materno
   9. nombre_completo
  10. rfc
  11. fecha_nacimiento
  12. edad
  13. sexo
  14. estado_civil
  15. ocupacion
  16. nivel_educacion
  17. ramo
  18. plan
  19. fecha_emision
  20. fecha_inicio_vigencia
  21. fecha_fin_vigencia
  22. num_renovaciones
  23. status_poliza
  24. motivo_baja
  25. canal_venta
  26. marca_vehiculo
  27. modelo_vehiculo
  28. tipo_vehiculo
  29. suma_asegurada
  30. deducible
  31. prima_neta
  32. prima_total
  33. cuota_prima
  34. forma_pago
  35. num_cuotas
  36. agente_id
  37. estado
  38. municipio
  39. codigo_postal
  40. coord_lat
  41. coord_lon
  42. version_documento
  43. hash_documento
  44. timestamp_carga
  45. usuario_captura
  46. ip_carga


In [8]:
# ── Paso 2: Diagnostico de todas las columnas ────────────────────────────────
# Cargamos todo para el diagnostico inicial
df_full = pd.read_csv('datos/cartera_polizas.csv')
mb_full = df_full.memory_usage(deep=True).sum() / 1024**2 # vemos cuanto utiliza el df en memoria
print(f'Dataset completo: {df_full.shape} · {mb_full:.1f} MB')
print()

# Perfil de cada columna: tipo, NaN%, valores unicos
print(f'{"Columna":<30} {"Dtype":<12} {"NaN%":>7} {"Unicos":>8}')
print('-' * 65)
for col in df_full.columns:
    dtype = str(df_full[col].dtype)
    nan_pct = df_full[col].isna().mean() * 100
    n_uniq  = df_full[col].nunique()
    print(f'{col:<30} {dtype:<12} {nan_pct:>6.1f}% {n_uniq:>8,}')

Dataset completo: (50000, 46) · 112.0 MB

Columna                        Dtype           NaN%   Unicos
-----------------------------------------------------------------
id_poliza                      object          0.0%   50,000
num_poliza                     object          0.0%   50,000
id_contrato_interno            object          0.0%   48,572
folio_emision                  object          0.0%   49,870
id_sistema_legacy              object          0.0%   49,988
nombre                         object          0.0%       55
apellido_paterno               object          0.0%       38
apellido_materno               object          0.0%       23
nombre_completo                object          0.0%   31,140
rfc                            object          0.0%   50,000
fecha_nacimiento               object          0.0%   15,676
edad                           int64           0.0%       46
sexo                           object          0.0%        4
estado_civil                   object 

In [4]:
# ── Paso 3: Clasificar las columnas por categoria ────────────────────────────
# Esta clasificacion es una DECISION DE NEGOCIO — no solo tecnica

ANALITICAS = [
    'id_poliza','num_poliza','ramo','plan','status_poliza',
    'nombre','apellido_paterno','apellido_materno',
    'fecha_nacimiento',
    'rfc','edad','sexo','estado_civil','ocupacion',
    'fecha_emision','fecha_inicio_vigencia','fecha_fin_vigencia',
    'num_renovaciones','motivo_baja',
    'suma_asegurada','deducible','prima_neta','prima_total',
    'forma_pago','agente_id','canal_venta',
    'estado','municipio','codigo_postal',
    'marca_vehiculo','modelo_vehiculo','tipo_vehiculo',  # solo Autos
]

REDUNDANTES = [
    'nombre_completo',        # = nombre + ap_pat + ap_mat
    'id_contrato_interno',    # ≈ id_poliza
    'folio_emision',          # ≈ num_poliza
    'cuota_prima',            # = prima_total / num_cuotas
    'num_cuotas',             # derivado de forma_pago
]

ADMINISTRATIVAS = [
    'hash_documento',         # hash SHA del PDF — auditoria IT
    'timestamp_carga',        # mismo valor para todos
    'ip_carga',               # IP del servidor batch
    'usuario_captura',        # operacion interna
    'version_documento',      # version del formato del contrato
    'id_sistema_legacy',      # util SOLO para joins con sistema core
]

CONDICIONALES = [
    'nivel_educacion',        # si lo necesitas para el modelo
    'coord_lat','coord_lon',  # solo si haces analisis geoespacial
]

print(f'Analiticas:     {len(ANALITICAS)}')
print(f'Redundantes:    {len(REDUNDANTES)}')
print(f'Administrativas:{len(ADMINISTRATIVAS)}')
print(f'Condicionales:  {len(CONDICIONALES)}')
print(f'Total:          {len(ANALITICAS)+len(REDUNDANTES)+len(ADMINISTRATIVAS)+len(CONDICIONALES)}')

Analiticas:     32
Redundantes:    5
Administrativas:6
Condicionales:  3
Total:          46


In [11]:
# ── Paso 4: Cargar SOLO las columnas que necesitamos ─────────────────────────
t0 = time.time()
df = pd.read_csv(
    'datos/cartera_polizas.csv',
    usecols=ANALITICAS, #cargamos solo las columnas que vamos a utilizar
    na_values=['N/D','N/A','ND','--','Sin dato',''],
)
t1 = time.time()
mb_opt = df.memory_usage(deep=True).sum() / 1024**2

print('=== COMPARATIVA DE CARGA ===')
print(f'Carga completa (46 cols):  {mb_full:.1f} MB')
print(f'Carga selectiva ({len(ANALITICAS)} cols):  {mb_opt:.1f} MB')
print(f'Reduccion de memoria:      {(1-mb_opt/mb_full)*100:.0f}%')
print(f'Tiempo de carga:           {(t1-t0)*1000:.0f} ms')
print()
print(f'Dataset de trabajo: {df.shape}')
df.head(3)

=== COMPARATIVA DE CARGA ===
Carga completa (46 cols):  112.0 MB
Carga selectiva (32 cols):  73.7 MB
Reduccion de memoria:      34%
Tiempo de carga:           864 ms

Dataset de trabajo: (50000, 32)


,id_poliza,num_poliza,nombre,apellido_paterno,apellido_materno,rfc,fecha_nacimiento,edad,sexo,estado_civil,...,tipo_vehiculo,suma_asegurada,deducible,prima_neta,prima_total,forma_pago,agente_id,estado,municipio,codigo_postal
0,POL-000001,Vid-21-000001,Gabriela,Moreno,Vega,MOGV020429CG6,29/04/2002,24,F,Union libre,...,NaN,3000000,NaN,54000.0,67651.20,Mensual,AG054,Veracruz,Poza Rica,36619.0
1,POL-000002,Aut-19-000002,Valeria,Torres,Castillo,TOVC020815IA8,15/08/2002,23,Femenino,Casado,...,Compacto,150000,8000.0,5250.0,6394.50,Trimestral,AG004,Michoacan,Morelia,58889.0
2,POL-000003,GMM-22-000003,Fernanda,Ramos,Silva,RAFS941018BC1,18/10/1994,31,M,Union libre,...,NaN,800000,5000.0,17600.0,22049.28,Mensual,AG051,Baja California,Tecate,45784.0


### 📝 Ejercicio 1 — Auditoria de columnas (8 min)

Usando el perfil que generaste arriba:
- **1a.** Identifica las 3 columnas con mas NaN — ¿tienen sentido esos NaN o son errores?
- **1b.** Encuentra columnas con menos de 5 valores unicos — ¿cuales deberian ser `category`?
- **1c.** Hay una columna numerica con un rango imposible (negativo o muy alto) — ¿cual es?
  *Pista: usa `df.describe()` sobre las columnas analiticas*

In [6]:
# Tu codigo aqui:
df.isna().sum().sort_values(ascending=False).head(3)

motivo_baja        47535
tipo_vehiculo      35248
modelo_vehiculo    35248
dtype: int64

In [7]:
df.nunique()[df.nunique() < 5]

sexo             4
ramo             4
status_poliza    4
motivo_baja      4
forma_pago       4
dtype: int64

In [9]:
df.describe().round(2)

,edad,num_renovaciones,modelo_vehiculo,suma_asegurada,deducible,prima_neta,prima_total,codigo_postal
count,50000.00,50000.00,14752.00,50000.00,42490.00,49000.00,50000.00,48500.00
mean,43.32,1.24,2020.00,1002000.00,8266.82,24419.54,29283.17,54947.45
std,12.96,1.13,3.17,1105162.53,5503.03,25474.13,30581.86,26072.92
min,21.00,0.00,2015.00,150000.00,1000.00,2400.00,2784.00,10000.00
25%,32.00,0.00,2017.00,300000.00,5000.00,6798.00,8120.00,32198.75
50%,43.00,1.00,2020.00,500000.00,8000.00,14000.00,17052.00,54997.00
75%,54.00,2.00,2023.00,1000000.00,10000.00,28000.00,33825.60,77475.00
max,66.00,4.00,2025.00,5000000.00,20000.00,145800.00,182658.24,99998.00


---
### Duda 2: Texto Sucio — str Operations

El campo `sexo` tiene 4 representaciones distintas del mismo valor.
Si no lo normalizas, `groupby('sexo')` produce 8 grupos en lugar de 2.

In [12]:
# ── Ver el problema ──────────────────────────────────────────────────────────
print('Valores unicos en sexo ANTES de limpiar:')
print(df['sexo'].value_counts(dropna=False))
print(f'Total valores distintos: {df["sexo"].nunique()}')
print()

# Si haces groupby ahora, obtienes grupos incorrectos:
grupos_incorrectos = df.groupby('sexo')['prima_total'].mean()
print(grupos_incorrectos.to_string())
print(f'Grupos sin limpiar: {len(grupos_incorrectos)} (deberian ser 2)')

Valores unicos en sexo ANTES de limpiar:
sexo
M            16697
F            16585
Femenino      8466
Masculino     8252
Name: count, dtype: int64
Total valores distintos: 4

sexo
F            29053.277976
Femenino     29305.291745
M            29457.558936
Masculino    29369.674515
Grupos sin limpiar: 4 (deberian ser 2)


In [ ]:
# ── Solucion: normalizar con str operations ──────────────────────────────────

# Paso 1: normalizar a mayusculas sin espacios
df['sexo'] = df['sexo'].str.strip().str.upper() # strip es para quitar espacios, 

# Paso 2: mapear todas las variantes al estandar
MAPA_SEXO = {
    'M': 'M', 'MASCULINO': 'M', 'HOMBRE': 'M', 'MASC': 'M',
    'F': 'F', 'FEMENINO': 'F', 'MUJER': 'F', 'FEM': 'F',
}
df['sexo'] = df['sexo'].map(MAPA_SEXO) # mapear las variables del campo sexo por lo que indica el diccionario
# es para homogeneizar 
# Valores no reconocidos quedan como NaN automaticamente

print('DESPUES de normalizar:')
print(df['sexo'].value_counts(dropna=False))
print()
# Ahora groupby correcto
print('Prima promedio por sexo:')
print(df.groupby('sexo')['prima_total'].mean().round(2))

DESPUES de normalizar:
sexo
F    25051
M    24949
Name: count, dtype: int64

Prima promedio por sexo:
sexo
F    29138.45
M    29428.49
Name: prima_total, dtype: float64


In [14]:
# ── Mas str operations sobre los datos reales ────────────────────────────────

# Limpiar codigo_postal: eliminar 'N/D' (ya convertido a NaN por na_values)
# Verificar:
print(f'CP con NaN: {df["codigo_postal"].isna().sum()}')
# Rellenar CP desconocido con 'DESCONOCIDO' para no perder la fila
df['codigo_postal'] = df['codigo_postal'].fillna('DESCONOCIDO')



CP con NaN: 1500


In [ ]:
# Extraer ramo y anio desde num_poliza ('GMM-24-000123')
df['ramo_codigo']  = df['num_poliza'].str.extract(r'^([A-Z]+)-')# etraer todo lo que es letra
# solo trae letras mayusculas
df['anio_poliza']  = df['num_poliza'].str.extract(r'-([0-9]{2})-').astype(float).astype('Int16')#exraer lo que es numero, y quedarse 
#con la segunda posicion. Regex no es 0 index, inicia en 1.

# Verificar
print(df[['num_poliza','ramo_codigo','anio_poliza']].head(5).to_string(index=False))

   num_poliza ramo_codigo  anio_poliza
Vid-21-000001         NaN           21
Aut-19-000002         NaN           19
GMM-22-000003         GMM           22
Vid-19-000004         NaN           19
Vid-20-000005         NaN           20


---
### Duda 3: Fechas como Texto — pd.to_datetime()

In [15]:
# ── El problema: fechas llegaron como strings ────────────────────────────────
print('Tipo ANTES de convertir:', df['fecha_nacimiento'].dtype)
print('Muestra:', df['fecha_nacimiento'].head(3).values)
print()



Tipo ANTES de convertir: object
Muestra: ['29/04/2002' '15/08/2002' '18/10/1994']



In [11]:
# ── Convertir fecha_nacimiento (formato d/m/Y) ────────────────────────────────
df['fecha_nacimiento'] = pd.to_datetime(
    df['fecha_nacimiento'],
    format='%d/%m/%Y',
    errors='coerce'  # fechas invalidas → NaT (no detiene el proceso)
)

# ── Convertir fechas ISO estandar ────────────────────────────────────────────
for col_fecha in ['fecha_emision','fecha_inicio_vigencia','fecha_fin_vigencia']:
    df[col_fecha] = pd.to_datetime(df[col_fecha], errors='coerce')

print('Fechas convertidas:')
print(df[['fecha_nacimiento','fecha_emision','fecha_fin_vigencia']].dtypes)
print(f'NaT en fecha_nacimiento: {df["fecha_nacimiento"].isna().sum()}')

Fechas convertidas:
fecha_nacimiento      datetime64[ns]
fecha_emision         datetime64[ns]
fecha_fin_vigencia    datetime64[ns]
dtype: object
NaT en fecha_nacimiento: 0


In [13]:
# ── Calculos actuariales con las fechas convertidas ──────────────────────────
hoy = pd.Timestamp.today()

# Edad calculada (mas precisa que la columna 'edad' del CSV)
df['edad_calc'] = ((hoy - df['fecha_nacimiento']).dt.days / 365.25)#.astype('Int8')

# Dias de vigencia de la poliza
df['dias_vigencia'] = (df['fecha_fin_vigencia'] - df['fecha_inicio_vigencia']).dt.days

# Fraccion expuesta (cuanto del periodo ya transcurrio)
dias_transcurridos = (hoy - df['fecha_inicio_vigencia']).dt.days
df['fraccion_expuesta'] = (dias_transcurridos / df['dias_vigencia']).clip(0, 1).round(4)

# Componentes de fecha para groupby temporal
df['anio_emision']     = df['fecha_emision'].dt.year
df['mes_emision']      = df['fecha_emision'].dt.month
df['trimestre_emision']= df['fecha_emision'].dt.quarter

# Verificar
print(df[['nombre','edad','edad_calc','dias_vigencia','fraccion_expuesta']].head(5).to_string(index=False))

  nombre  edad  edad_calc  dias_vigencia  fraccion_expuesta
Gabriela    24  24.008214            365                1.0
 Valeria    23  23.712526            366                1.0
Fernanda    31  31.537303            365                1.0
  Silvia    40  40.238193            366                1.0
 Antonio    29  29.776865            365                1.0


### 📝 Ejercicio 2 — Limpiar siniestros.csv (10 min)

El archivo `siniestros.csv` tiene fechas en 3 formatos distintos:
- `fecha_ocurrencia`: YYYY-MM-DD
- `fecha_apertura`: d/m/Y (mismo dia que fecha_reporte pero formato distinto — es REDUNDANTE)
- `fecha_ultimo_movimiento`: d/m/Y

**2a.** Carga siniestros.csv usando SOLO las columnas utiles (descarta las administrativas y redundantes).
**2b.** Convierte las 3 columnas de fecha a datetime con el formato correcto.
**2c.** Calcula `dias_reporte_real` = fecha_reporte - fecha_ocurrencia.
**2d.** Calcula `dias_resolucion_real` = fecha_cierre - fecha_reporte (NaT si no esta cerrado).
**2e.** Verifica: ¿cuantos siniestros llevan mas de 180 dias sin cerrar?

In [15]:
# Tu codigo aqui:
siniestros_cols_utiles = [
    'id_siniestro','id_poliza','ramo','tipo_siniestro', 'fecha_apertura',
    'fecha_ocurrencia','fecha_reporte','fecha_ultimo_movimiento','fecha_cierre',
    'dias_reporte','monto_reclamado','monto_pagado',
    'status_siniestro','motivo_rechazo','id_ajustador',
]
# Carga, convierte fechas y calcula los campos derivados:
df2 = pd.read_csv(
    'datos/siniestros.csv',
    usecols=siniestros_cols_utiles, #cargamos solo las columnas que vamos a utilizar
    na_values=['N/D','N/A','ND','--','Sin dato',''],
)

In [18]:
for col_fecha in ['fecha_ocurrencia','fecha_apertura','fecha_ultimo_movimiento', 'fecha_reporte', 'fecha_cierre']:
    df2[col_fecha] = pd.to_datetime(df2[col_fecha], errors='coerce')

print('Fechas convertidas:')
print(df2[['fecha_ocurrencia','fecha_apertura','fecha_ultimo_movimiento', 'fecha_reporte', 'fecha_cierre']].dtypes)

Fechas convertidas:
fecha_ocurrencia           datetime64[ns]
fecha_apertura             datetime64[ns]
fecha_ultimo_movimiento    datetime64[ns]
fecha_reporte              datetime64[ns]
fecha_cierre               datetime64[ns]
dtype: object


In [19]:
df2['dias_reporte_real'] = (df2['fecha_reporte'] - df2['fecha_ocurrencia']).dt.days

In [ ]:
df2['dias_solucion_real'] = (df2['fecha_cierre'] - df2['fecha_reporte']).dt.days

---
### Duda 4: JSON — Datos de API


In [20]:
# ── Simular una respuesta JSON de una API de reaseguro ───────────────────────
import json

# Este es el tipo de JSON que recibirias de una API REST
respuesta_api = {
    'timestamp': '2026-05-02T07:00:00',
    'origen': 'Sistema_Reaseguro_v3.1',
    'polizas_reaseguradas': [
        {'id_poliza':'POL-000001','ramo':'GMM',
         'reasegurador':{'nombre':'Munich Re','participacion':0.40,'prima':960.0},
         'limites':{'maximo':2_000_000,'retencion':500_000}},
        {'id_poliza':'POL-000002','ramo':'Vida',
         'reasegurador':{'nombre':'Swiss Re','participacion':0.35,'prima':1820.0},
         'limites':{'maximo':5_000_000,'retencion':1_000_000}},
        {'id_poliza':'POL-000005','ramo':'GMM',
         'reasegurador':{'nombre':'Munich Re','participacion':0.40,'prima':550.0},
         'limites':{'maximo':2_000_000,'retencion':500_000}},
    ]
}

# Guardar como JSON (simula lo que llegaria de la API)
with open('datos/respuesta_reaseguro.json','w',encoding='utf-8') as f:
    json.dump(respuesta_api, f, ensure_ascii=False, indent=2)

print('JSON guardado en datos/respuesta_reaseguro.json')
print('Primeras lineas:')
print(json.dumps(respuesta_api, indent=2, ensure_ascii=False)[:300])

JSON guardado en datos/respuesta_reaseguro.json
Primeras lineas:
{
  "timestamp": "2026-05-02T07:00:00",
  "origen": "Sistema_Reaseguro_v3.1",
  "polizas_reaseguradas": [
    {
      "id_poliza": "POL-000001",
      "ramo": "GMM",
      "reasegurador": {
        "nombre": "Munich Re",
        "participacion": 0.4,
        "prima": 960.0
      },
      "limites": 


In [21]:
# ── Problema: JSON anidado no se carga directo en DataFrame ─────────────────
# pd.read_json funciona para JSON simple pero no para estructuras anidadas

# Cargar el JSON, forma más usual. JSON: bases de datos no relacionales
with open('datos/respuesta_reaseguro.json') as f:
    data = json.load(f)

# Intentar con pd.read_json — no da el resultado esperado con anidados
# pd.read_json('datos/respuesta_reaseguro.json')  # solo lee nivel 1

# ── Solucion: json_normalize aplana el JSON anidado ──────────────────────────
from pandas import json_normalize

df_reas = json_normalize(
    data['polizas_reaseguradas'],
    sep='_'    # separador para campos anidados
)

print('DataFrame aplanado:')
print(df_reas.to_string(index=False))
print()
print('Columnas generadas:', list(df_reas.columns))

DataFrame aplanado:
 id_poliza ramo reasegurador_nombre  reasegurador_participacion  reasegurador_prima  limites_maximo  limites_retencion
POL-000001  GMM           Munich Re                        0.40               960.0         2000000             500000
POL-000002 Vida            Swiss Re                        0.35              1820.0         5000000            1000000
POL-000005  GMM           Munich Re                        0.40               550.0         2000000             500000

Columnas generadas: ['id_poliza', 'ramo', 'reasegurador_nombre', 'reasegurador_participacion', 'reasegurador_prima', 'limites_maximo', 'limites_retencion']


In [ ]:
# ── json_normalize con listas anidadas ───────────────────────────────────────
# Caso mas complejo: cuando hay listas dentro del JSON

respuesta_multi = {
    'polizas': [
        {'id':'P01','coberturas':[{'tipo':'Hospitalizacion','suma':500_000},{'tipo':'Cirugia','suma':300_000}]},
        {'id':'P02','coberturas':[{'tipo':'Hospitalizacion','suma':800_000}]},
    ]
}

# record_path: donde esta la lista a 'explotar'
# meta: campos del nivel padre que quieres conservar
df_cob = json_normalize(
    respuesta_multi['polizas'],
    record_path='coberturas',
    meta=['id'],
    sep='_'
)
print(df_cob)

In [ ]:
# ── Guardar DataFrame como JSON ──────────────────────────────────────────────

# orient='records' — lista de objetos (lo mas comun para APIs)
df.head(10).to_json('datos/muestra.json',
    orient='records',
    force_ascii=False,  # preserva caracteres especiales (acentos)
    indent=2,
    date_format='iso'   # fechas en formato ISO
)

# Verificar
with open('datos/muestra.json') as f:
    preview = f.read(300)
print(preview)

---
### Duda 5: 90k Filas — Procesar sin Esperar (chunks)

In [16]:
# ── Ver el tamano del archivo de beneficiarios ───────────────────────────────
mb_ben = os.path.getsize('datos/beneficiarios.csv') / 1024**2
print(f'beneficiarios.csv: {mb_ben:.1f} MB')



beneficiarios.csv: 11.5 MB


In [17]:
# Contar filas sin cargar todo
total_ben = sum(len(chunk) for chunk in
    pd.read_csv('datos/beneficiarios.csv', chunksize=10_000))
print(f'Total beneficiarios: {total_ben:,}')

# ── Patron real: calcular estadisticas por parentesco ─────────────────────────
# Sin cargar los 90k en memoria a la vez
conteo_parentesco = {}

for chunk in pd.read_csv('datos/beneficiarios.csv', chunksize=10_000):
    counts = chunk['parentesco'].value_counts().to_dict()
    for k, v in counts.items():
        conteo_parentesco[k] = conteo_parentesco.get(k, 0) + v

resultado = pd.Series(conteo_parentesco).sort_values(ascending=False)
print('Beneficiarios por parentesco (procesado por chunks):')
print(resultado)

Total beneficiarios: 90,000
Beneficiarios por parentesco (procesado por chunks):
Conyuge    26718
Hijo       22785
Hija       17871
Madre       7298
Padre       7173
Hermano     3596
Hermana     2765
Otro        1794
dtype: int64


In [24]:
# ── Filtrar y concatenar solo lo que necesitas ───────────────────────────────
# Obtener solo beneficiarios activos de polizas de Vida

partes = []
for chunk in pd.read_csv('datos/beneficiarios.csv',
                         chunksize=10_000,
                         usecols=['id_beneficiario','id_poliza','nombre',
                                  'apellido_paterno','parentesco','porcentaje','activo']):
    filtrado = chunk[chunk['activo'] == True]
    if len(filtrado) > 0:
        partes.append(filtrado)

ben_activos = pd.concat(partes, ignore_index=True)
print(f'Beneficiarios activos: {len(ben_activos):,}')
print(f'Memoria: {ben_activos.memory_usage(deep=True).sum()/1024:.0f} KB')

Beneficiarios activos: 67,656
Memoria: 21956 KB


---
### Duda 6: Downcast — Cuándo Es Seguro

In [25]:
# ── Demostrar el problema de precision ───────────────────────────────────────
import numpy as np

# float64 vs float32 con valores grandes
prima_grande = 15_432_756.80
print(f'Original (float64): {prima_grande}')
print(f'Como float32:       {np.float32(prima_grande)}')
print(f'Diferencia:         {abs(prima_grande - float(np.float32(prima_grande))):.2f}')
print()

# Con valores tipicos de primas individuales
prima_gmm = 3_450.75
print(f'Prima GMM (float64): {prima_gmm}')
print(f'Prima GMM (float32): {np.float32(prima_gmm)}')
print(f'Diferencia:          {abs(prima_gmm - float(np.float32(prima_gmm))):.6f}')

Original (float64): 15432756.8
Como float32:       15432757.0
Diferencia:         0.20

Prima GMM (float64): 3450.75
Prima GMM (float32): 3450.75
Diferencia:          0.000000


In [26]:
# ── Estrategia segura de optimizacion ────────────────────────────────────────
print('Memoria ANTES de optimizar:')
print(f'{df.memory_usage(deep=True).sum()/1024**2:.2f} MB')
print()

df_opt = df.copy()

# SEGURO: categoricas para columnas con pocos valores unicos
cols_category = ['ramo','plan','status_poliza','sexo','canal_venta',
                 'forma_pago','estado','estado_civil','tipo_vehiculo']
for col in cols_category:
    if col in df_opt.columns:
        n_uniq = df_opt[col].nunique()
        n_tot  = len(df_opt)
        pct = n_uniq/n_tot
        print(f'  {col:<25}: {n_uniq:>5} unicos ({pct:.1%}) → category')
        df_opt[col] = df_opt[col].astype('category')

# SEGURO: enteros pequenos
df_opt['num_renovaciones'] = df_opt['num_renovaciones'].fillna(0).astype('int8')

# SEGURO: booleano
# (activa ya no esta porque la descartamos, pero aplica el principio)

# CONSERVAR en float64: primas y sumas (montos grandes o con centavos importantes)
# NO hacer: df_opt['prima_total'] = df_opt['prima_total'].astype('float32')

print()
print('Memoria DESPUES de optimizar:')
mb_antes = df.memory_usage(deep=True).sum()/1024**2
mb_desp  = df_opt.memory_usage(deep=True).sum()/1024**2
print(f'{mb_antes:.2f} MB → {mb_desp:.2f} MB ({(1-mb_desp/mb_antes)*100:.0f}% reduccion)')

Memoria ANTES de optimizar:
65.24 MB

  ramo                     :     4 unicos (0.0%) → category
  plan                     :    12 unicos (0.0%) → category
  status_poliza            :     4 unicos (0.0%) → category
  sexo                     :     2 unicos (0.0%) → category
  canal_venta              :     6 unicos (0.0%) → category
  forma_pago               :     4 unicos (0.0%) → category
  estado                   :    15 unicos (0.0%) → category
  estado_civil             :     5 unicos (0.0%) → category
  tipo_vehiculo            :     8 unicos (0.0%) → category

Memoria DESPUES de optimizar:
65.24 MB → 39.18 MB (40% reduccion)


### 📝 Ejercicio 3 — Pipeline de limpieza completo (12 min)

Encapsula todo lo que hicimos en una funcion `limpiar_cartera(ruta_csv)` que:
- Carga con `usecols=ANALITICAS`
- Normaliza `sexo` con str + map
- Convierte fechas con `to_datetime` + `errors='coerce'`
- Calcula `edad_calc`, `dias_vigencia`, `fraccion_expuesta`
- Aplica optimizacion de memoria (categoricas)
- Retorna el DataFrame limpio

Al final llama: `df_limpio = limpiar_cartera('datos/cartera_polizas.csv')`
y verifica que no tenga texto sucio en sexo.

In [ ]:
# Tu codigo aqui:
def limpiar_cartera(ruta_csv):
    """Leer un csv  """
    # Implementa la funcion completa aqui
    # Variables a utilizar
    ANALITICAS = [
    'id_poliza','num_poliza','ramo','plan','status_poliza',
    'nombre','apellido_paterno','apellido_materno',
    'fecha_nacimiento',
    'rfc','edad','sexo','estado_civil','ocupacion',
    'fecha_emision','fecha_inicio_vigencia','fecha_fin_vigencia',
    'num_renovaciones','motivo_baja',
    'suma_asegurada','deducible','prima_neta','prima_total',
    'forma_pago','agente_id','canal_venta',
    'estado','municipio','codigo_postal',
    'marca_vehiculo','modelo_vehiculo','tipo_vehiculo',  # solo Autos
    ]
    # leemos el csv
    df = pd.read_csv(
    ruta_csv,
    usecols=ANALITICAS, #cargamos solo las columnas que vamos a utilizar
    na_values=['N/D','N/A','ND','--','Sin dato',''],
    )
    # Empezamos a 'limpiar'
    df['sexo'] = df['sexo'].str.strip().str.upper() # strip es para quitar espacios, 

    
    MAPA_SEXO = {
        'M': 'M', 'MASCULINO': 'M', 'HOMBRE': 'M', 'MASC': 'M',
        'F': 'F', 'FEMENINO': 'F', 'MUJER': 'F', 'FEM': 'F',
    }
    df['sexo'] = df['sexo'].map(MAPA_SEXO)

    #convertimos las fechas a variables de tipo fecha
    for col_fechas in ['fecha_emision', 'fecha_inicio_vigencia', 'fecha_fin_vigencia']:
        df[col_fecha] = pd.to_datetime(df[col_fecha], errors='coerce')

    hoy = pd.Timestamp.today()

    # Edad calculada (mas precisa que la columna 'edad' del CSV)
    df['edad_calc'] = ((hoy - df['fecha_nacimiento']).dt.days / 365.25)#.astype('Int8')

    # Dias de vigencia de la poliza
    df['dias_vigencia'] = (df['fecha_fin_vigencia'] - df['fecha_inicio_vigencia']).dt.days

    # Fraccion expuesta (cuanto del periodo ya transcurrio)
    dias_transcurridos = (hoy - df['fecha_inicio_vigencia']).dt.days
    df['fraccion_expuesta'] = (dias_transcurridos / df['dias_vigencia']).clip(0, 1).round(4)

    df_opt = df.copy()

    # SEGURO: categoricas para columnas con pocos valores unicos
    cols_category = ['ramo','plan','status_poliza','sexo','canal_venta',
                    'forma_pago','estado','estado_civil','tipo_vehiculo']
    for col in cols_category:
        if col in df_opt.columns:
            n_uniq = df_opt[col].nunique()
            n_tot  = len(df_opt)
            df_opt[col] = df_opt[col].astype('category')
    
    return df_opt
    pass


---
### Duda 7: ¿Cuándo Cambiar a Polars?

In [18]:
# ── Instalar polars si no esta disponible ────────────────────────────────────
# En tu terminal: pip install polars
# Verificar:
try:
    import polars as pl
    print(f'Polars {pl.__version__} disponible')
    POLARS_OK = True
except ImportError:
    print('Polars no instalado. Ejecuta: pip install polars')
    POLARS_OK = False

Polars 1.40.1 disponible


In [28]:
!pip install polars

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 828.7/828.7 kB 6.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.1/52.1 MB 10.2 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [polars]2m1/2 [polars]


In [8]:
# ── Comparativa de velocidad pandas vs polars ────────────────────────────────
if POLARS_OK:
    import polars as pl
    import time
    import pandas as pd

    # ── Pandas ──────────────────────────────────────────────────────────────
    t0 = time.time()
    df_pd = pd.read_csv('datos/cartera_polizas.csv', usecols=ANALITICAS)
    res_pd = (df_pd.groupby('ramo')
                   .agg(polizas=('id_poliza','count'),
                        prima_total=('prima_total','sum'),
                        prima_prom=('prima_neta','mean'))
                   .round(2).reset_index())
    t_pd = time.time()-t0

    # ── Polars ──────────────────────────────────────────────────────────────
    t0 = time.time()
    df_pl = pl.read_csv('datos/cartera_polizas.csv', columns=ANALITICAS)
    res_pl = (df_pl
        .group_by('ramo')
        .agg([
            pl.col('id_poliza').count().alias('polizas'),
            pl.col('prima_total').sum().alias('prima_total'),
            pl.col('prima_neta').mean().alias('prima_prom'),
        ])
    )
    t_pl = time.time()-t0

    print(f'Pandas:  {t_pd*1000:.0f} ms')
    print(f'Polars:  {t_pl*1000:.0f} ms')
    print(f'Polars es {t_pd/t_pl:.1f}x mas rapido en esta operacion')
    print()
    print('Resultado Polars:')
    print(res_pl)
else:
    print('Instala polars para ver la comparativa: pip install polars')

Pandas:  1412 ms
Polars:  90 ms
Polars es 15.7x mas rapido en esta operacion

Resultado Polars:
shape: (4, 4)
┌───────────────────────┬─────────┬─────────────┬──────────────┐
│ ramo                  ┆ polizas ┆ prima_total ┆ prima_prom   │
│ ---                   ┆ ---     ┆ ---         ┆ ---          │
│ str                   ┆ u32     ┆ f64         ┆ f64          │
╞═══════════════════════╪═════════╪═════════════╪══════════════╡
│ GMM                   ┆ 22531   ┆ 7.3102e8    ┆ 27065.47908  │
│ Accidentes Personales ┆ 5207    ┆ 2.4981e7    ┆ 4002.032838  │
│ Vida                  ┆ 7510    ┆ 4.5433e8    ┆ 50439.777113 │
│ Autos                 ┆ 14752   ┆ 2.5382e8    ┆ 14349.345658 │
└───────────────────────┴─────────┴─────────────┴──────────────┘


In [6]:
# ── Sintaxis Polars: las operaciones mas comunes ──────────────────────────────
if POLARS_OK:
    df_pl = pl.read_csv('datos/cartera_polizas.csv',
                        columns=['id_poliza','ramo','prima_total','edad'])

    # Filtrar
    print('Polizas con prima > 10,000:')
    print(df_pl.filter(pl.col('prima_total') > 10_000).shape)

    # Agregar columna
    df_pl2 = df_pl.with_columns(
        (pl.col('prima_total')/12).alias('prima_mensual'),
        #(pl.col('siniest') >= 2).alias('alto_riesgo'),
    )

    # Sort
    print('Top 5 primas:')
    print(df_pl2.sort('prima_total', descending=True).head(5)[['id_poliza','ramo','prima_total']])

    # Lazy evaluation — ejecuta todo optimizado al final
    resultado = (
        pl.scan_csv('datos/cartera_polizas.csv')  # NO carga en memoria aun
        .filter(pl.col('prima_total') > 5000)
        .group_by('ramo')
        .agg(pl.col('prima_total').sum())
        .collect()  # AHORA ejecuta con el plan optimizado
    )
    print('Lazy evaluation result:')
    print(resultado)

Polizas con prima > 10,000:
(33919, 4)
Top 5 primas:
shape: (5, 3)
┌────────────┬──────┬─────────────┐
│ id_poliza  ┆ ramo ┆ prima_total │
│ ---        ┆ ---  ┆ ---         │
│ str        ┆ str  ┆ f64         │
╞════════════╪══════╪═════════════╡
│ POL-007740 ┆ Vida ┆ 182658.24   │
│ POL-007890 ┆ Vida ┆ 182658.24   │
│ POL-024748 ┆ Vida ┆ 182658.24   │
│ POL-003193 ┆ Vida ┆ 180403.2    │
│ POL-007981 ┆ Vida ┆ 180403.2    │
└────────────┴──────┴─────────────┘
Lazy evaluation result:
shape: (4, 2)
┌───────────────────────┬─────────────┐
│ ramo                  ┆ prima_total │
│ ---                   ┆ ---         │
│ str                   ┆ f64         │
╞═══════════════════════╪═════════════╡
│ Vida                  ┆ 4.5433e8    │
│ Autos                 ┆ 2.5382e8    │
│ GMM                   ┆ 7.3102e8    │
│ Accidentes Personales ┆ 1.2546e7    │
└───────────────────────┴─────────────┘


### Regla practica: ¿Pandas o Polars?

| Situacion | Usa |
|-----------|-----|
| Aprendizaje, primeros modelos | **pandas** — el ecosistema es enorme |
| < 1 millon de filas | **pandas** — mas que suficiente |
| Analisis interactivo en notebook | **pandas** — syntax mas conocida |
| Pipeline de produccion > 5M filas | **polars** — 5-20x mas rapido |
| Ingesta diaria de datos grandes | **polars** con lazy evaluation |
| ETL de empresa | **polars** o **spark** segun el tamano |

---
## pivot_table con Datos Reales


In [5]:
# ── Agregar columnas necesarias para el pivot ────────────────────────────────
import pandas as pd
df_work = pd.read_csv('datos/cartera_polizas.csv', usecols=ANALITICAS,
                      na_values=['N/D','N/A',''])
df_work['prima_neta'] = df_work['prima_neta'].fillna(df_work.groupby('ramo')['prima_neta'].transform('median'))
df_work['g_edad'] = pd.cut(df_work['edad'], bins=[0,30,45,60,100],
                           labels=['18-30','31-45','46-60','61+'])
df_work['siniest_flag'] = (df_work['prima_neta'] > df_work['prima_neta'].quantile(0.75)).astype(int)

print(f'Dataset para pivot: {df_work.shape}')

Dataset para pivot: (50000, 34)


In [10]:
# ── pivot_table: prima por ramo x grupo de edad ──────────────────────────────
tabla_prima = pd.pivot_table(
    df_work,
    values   = 'prima_total',
    index    = 'ramo',
    columns  = 'g_edad',
    aggfunc  = 'sum',
    fill_value = 0,
    margins    = True,
    margins_name = 'TOTAL'
).round(0) / 1_000  # en miles de pesos

print('Prima total por ramo y grupo de edad (miles MXN):')
print(tabla_prima.to_string())

Prima total por ramo y grupo de edad (miles MXN):
g_edad                      18-30       31-45       46-60         61+        TOTAL
ramo                                                                              
Accidentes Personales    5248.369    8348.395    8731.668    2652.247    24980.679
Autos                   55457.022   84225.512   84571.404   29569.569   253823.506
GMM                    139278.258  223908.688  264218.298  103617.078   731022.322
Vida                    79667.849  134774.641  166301.773   73587.860   454332.123
TOTAL                  279651.498  451257.236  523823.143  209426.754  1464158.631


/var/folders/k1/_jflqvn90v31fwxjw3lf6y680000gn/T/ipykernel_79661/3938379720.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  tabla_prima = pd.pivot_table(


In [11]:
# ── pivot_table: polizas por zona x canal ────────────────────────────────────
tabla_canal = pd.pivot_table(
    df_work,
    values   = 'id_poliza',
    index    = 'estado',
    columns  = 'canal_venta',
    aggfunc  = 'count',
    fill_value = 0,
    margins    = True,
    margins_name = 'TOTAL'
)

print('Polizas por estado y canal de venta:')
print(tabla_canal.to_string())

Polizas por estado y canal de venta:
canal_venta       Agente  Banca Seguros  Broker  Digital  Directo  Promotor  TOTAL
estado                                                                            
Baja California     1646            329     678      241      356        98   3348
CDMX                1719            311     684      217      318       108   3357
Chihuahua           1656            344     642      250      357       101   3350
Coahuila            1721            349     672      199      329        96   3366
Estado de Mexico    1703            336     652      253      287        93   3324
Guanajuato          1745            339     669      237      359        96   3445
Jalisco             1664            322     670      258      332       106   3352
Michoacan           1621            327     711      217      319        99   3294
Nuevo Leon          1681            335     615      225      326        96   3278
Puebla              1586            330     672   

---
## Exportar — CSV, Excel, Parquet y JSON


In [6]:
# ── Guardar en todos los formatos y comparar ──────────────────────────────────
import os, time
df_export = df_work.head(10_000)  # subconjunto para demo

formatos = {
    'CSV':     ('datos/export_demo.csv',
                lambda: df_export.to_csv('datos/export_demo.csv', index=False)),
    'Parquet': ('datos/export_demo.parquet',
                lambda: df_export.to_parquet('datos/export_demo.parquet', index=False)),
    'JSON':    ('datos/export_demo.json',
                lambda: df_export.to_json('datos/export_demo.json',
                                          orient='records', force_ascii=False)),
}

print(f'{"Formato":<10} {"Tamano":>10} {"Tiempo":>10}')
print('-' * 35)
for nombre, (ruta, guardar) in formatos.items():
    t0 = time.time()
    guardar()
    t = (time.time()-t0)*1000
    kb = os.path.getsize(ruta)/1024
    print(f'{nombre:<10} {kb:>8.0f} KB {t:>8.0f} ms')

Formato        Tamano     Tiempo
-----------------------------------
CSV            2455 KB      492 ms
Parquet         633 KB     5377 ms
JSON           7645 KB      140 ms


In [14]:
!pip install pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 10.4 MB/s  0:00:03m0:00:0100:01


In [15]:
!pip install fastparquet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 891.0/891.0 kB 7.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 12.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [fastparquet] [fastparquet]


In [ ]:
# ── Excel multihoja — el entregable mas solicitado en empresas ───────────────
resumen_ramo = df_work.groupby('ramo').agg(
    polizas    =('id_poliza','count'),
    prima_total=('prima_total','sum'),
    prima_prom =('prima_total','mean'),
).round(2).reset_index()
resumen_ramo['pct_cartera'] = (resumen_ramo['prima_total']/resumen_ramo['prima_total'].sum()*100).round(1)

with pd.ExcelWriter('datos/reporte_demo.xlsx', engine='openpyxl') as writer:
    df_work.head(5_000).to_excel(writer, sheet_name='Cartera', index=False)
    resumen_ramo.to_excel(writer, sheet_name='Resumen_Ramo', index=False)
    tabla_prima.to_excel(writer, sheet_name='Pivot_Prima')
    tabla_canal.to_excel(writer, sheet_name='Pivot_Canal')

kb_xl = os.path.getsize('datos/reporte_demo.xlsx')/1024
print(f'Excel multihoja generado: {kb_xl:.0f} KB con 4 hojas')

---
## 🏆 Ejercicio Integrador Final — Pipeline Completo

**Contexto:** Tu jefa de estadistica te pide el reporte ejecutivo del Q1 2026.
Tienes los 4 archivos crudos. Debes construir un pipeline completo,
documentando cada decision de limpieza.

**Tiempo:** 40 minutos  |  **Entregable:** Excel con 5 hojas + Parquet

---

### Criterios de evaluacion:
- ✅ Cada decision de descarte de columnas esta justificada con comentario
- ✅ No hay texto sucio en columnas categoricas clave
- ✅ Todas las fechas son datetime, no object
- ✅ dtypes optimizados sin perder precision en montos
- ✅ El Excel tiene las 5 hojas con los datos correctos
- ✅ El Parquet es mas pequeno que el CSV equivalente

In [24]:
# ══════════════════════════════════════════════════════════════════════════════
# FASE 1: INGESTA INTELIGENTE
# ══════════════════════════════════════════════════════════════════════════════
# Carga cartera, siniestros y catalogo de ramos/agentes.
# Usa SOLO las columnas que necesitas — justifica con comentario.
# Mide la memoria ahorrada vs cargar todo.

# Tu codigo aqui:
import pandas as pd
import numpy as np
import os, time
# Cargamos los insumos y analizaremos las columnas a ocupar
df = pd.read_csv('datos/cartera_polizas.csv')
mem_total = df.memory_usage(deep=True).sum() / 1024**2
siniestros = pd.read_csv('datos/siniestros.csv')
cat_ramos = pd.read_csv('datos/catalogo_ramos.csv')
cat_agentes = pd.read_csv('datos/catalogo_agentes.csv')

print(f'{"Columna":<30} {"Dtype":<12} {"NaN%":>7} {"Unicos":>8}')
for col in df.columns:
    dtype = str(df[col].dtype)
    nan_pct = df[col].isna().mean() * 100
    n_uniq  = df[col].nunique()
    print(f'{col:<30} {dtype:<12} {nan_pct:>6.1f}% {n_uniq:>8,}')

Columna                        Dtype           NaN%   Unicos
id_poliza                      object          0.0%   50,000
num_poliza                     object          0.0%   50,000
id_contrato_interno            object          0.0%   48,572
folio_emision                  object          0.0%   49,870
id_sistema_legacy              object          0.0%   49,988
nombre                         object          0.0%       55
apellido_paterno               object          0.0%       38
apellido_materno               object          0.0%       23
nombre_completo                object          0.0%   31,140
rfc                            object          0.0%   50,000
fecha_nacimiento               object          0.0%   15,676
edad                           int64           0.0%       46
sexo                           object          0.0%        4
estado_civil                   object          0.0%        5
ocupacion                      object          8.0%       20
nivel_educacion         

In [3]:
# Las variables a continuacion son las que no utilizaremos
NO_UTILES = [
    'nombre_completo',        # redudante ya que se conforma por nombre + ap_pat + ap_mat
    'id_contrato_interno',    # ya lo tenemos en id_poliza
    'folio_emision',          # ya lo tenemos en num_poliza
    'cuota_prima',            # campo calculado a partir de prima_total y num_cuotas
    'num_cuotas',             # derivado de forma_pago
    'hash_documento',         # hash SHA del PDF — auditoria IT
    'timestamp_carga',        # tiene el mismo valor para todos
    'ip_carga',               # IP del servidor batch
    'usuario_captura',        # operacion interna
    'version_documento',      # version del formato del contrato
    'id_sistema_legacy',      # util SOLO para joins con sistema core
    'nivel_educacion',        # no se analizará de momento
    'coord_lat','coord_lon',  # util para analisis geoespacial
]

In [25]:
UTILIZAR = [
    'id_poliza','num_poliza','ramo','plan','status_poliza',
    'nombre','apellido_paterno','apellido_materno',
    'fecha_nacimiento',
    'rfc','edad','sexo','estado_civil','ocupacion',
    'fecha_emision','fecha_inicio_vigencia','fecha_fin_vigencia',
    'num_renovaciones','motivo_baja',
    'suma_asegurada','deducible','prima_neta','prima_total',
    'forma_pago','agente_id','canal_venta',
    'estado','municipio','codigo_postal',
    'marca_vehiculo','modelo_vehiculo','tipo_vehiculo',  # solo Autos
]

In [26]:
# Cargamos la cartera solo con las variables a utilizar y medimos la memoria
df2 = pd.read_csv('datos/cartera_polizas.csv', usecols=UTILIZAR, na_values=['N/D','N/A','ND','--','Sin dato',''])
mem_df2 = df2.memory_usage(deep=True).sum() / 1024**2
print(f'Memoria utilizada cargando 46 columnas:  {mem_total:.1f} MB')
print(f'Memoria utilizada cargando {len(UTILIZAR)} columnas:  {mem_df2:.1f} MB')
print(f"Numero de registros: {len(df2)}")

Memoria utilizada cargando 46 columnas:  112.0 MB
Memoria utilizada cargando 32 columnas:  73.7 MB
Numero de registros: 50000


In [27]:

# ══════════════════════════════════════════════════════════════════════════════
# FASE 2: LIMPIEZA COMPLETA
# ══════════════════════════════════════════════════════════════════════════════
# 2a. Elimina duplicados de cartera
# 2b. Normaliza sexo con str.strip().str.upper() + .map(MAPA_SEXO)
# 2c. Convierte TODAS las fechas a datetime con errors='coerce'
# 2d. Rellena NaN de prima_neta con la MEDIANA POR RAMO (no global)
#     df.groupby('ramo')['prima_neta'].transform(lambda x: x.fillna(x.median()))
# 2e. Limpia codigo_postal (reemplaza 'N/D' con NaN)
# 2f. Aplica optimizacion de categoricas (sin tocar float64 de primas)

# Tu codigo aqui:

# Eliminamos duplicados
df2 = df2.drop_duplicates()

# Normalizamos texto (sexo, codigo_postal)
MAPA_SEXO = {
    'M': 'M', 'MASCULINO': 'M', 'HOMBRE': 'M', 'MASC': 'M',
    'F': 'F', 'FEMENINO': 'F', 'MUJER': 'F', 'FEM': 'F',
}
df2['sexo'] = df2['sexo'].str.strip().str.upper().map(MAPA_SEXO)

# Convirtimos fechas a datetime
if 'fecha_nacimiento' in df2.columns:
        df2['fecha_nacimiento'] = pd.to_datetime(
            df2['fecha_nacimiento'], format='%d/%m/%Y', errors='coerce')

    
for col in ['fecha_emision','fecha_inicio_vigencia','fecha_fin_vigencia']:
    if col in df2.columns:
        df2[col] = pd.to_datetime(df2[col], errors='coerce')

# Rellenamos NaN de prima_neta
df2['prima_neta'] = df2.groupby('ramo')['prima_neta'].transform(
    lambda x: x.fillna(x.median()))

# Limpiamos CP
df2['codigo_postal'] = df2['codigo_postal'].replace('N/D', np.nan)
    
df2_copy = df2.copy()

# optimizamos categorias
cols_category = ['ramo','plan','status_poliza','sexo','canal_venta',
                 'forma_pago','estado','estado_civil','tipo_vehiculo']
for col in cols_category:
    if col in df2_copy.columns:
        df2_copy[col] = df2_copy[col].astype('category')

print(F"Numero de registros: {len(df2_copy)}")

df2_copy.head(3)


Numero de registros: 50000


,id_poliza,num_poliza,nombre,apellido_paterno,apellido_materno,rfc,fecha_nacimiento,edad,sexo,estado_civil,...,tipo_vehiculo,suma_asegurada,deducible,prima_neta,prima_total,forma_pago,agente_id,estado,municipio,codigo_postal
0,POL-000001,Vid-21-000001,Gabriela,Moreno,Vega,MOGV020429CG6,2002-04-29,24,F,Union libre,...,NaN,3000000,NaN,54000.0,67651.20,Mensual,AG054,Veracruz,Poza Rica,36619.0
1,POL-000002,Aut-19-000002,Valeria,Torres,Castillo,TOVC020815IA8,2002-08-15,23,F,Casado,...,Compacto,150000,8000.0,5250.0,6394.50,Trimestral,AG004,Michoacan,Morelia,58889.0
2,POL-000003,GMM-22-000003,Fernanda,Ramos,Silva,RAFS941018BC1,1994-10-18,31,M,Union libre,...,NaN,800000,5000.0,17600.0,22049.28,Mensual,AG051,Baja California,Tecate,45784.0


In [29]:
df2_copy[['fecha_emision','fecha_inicio_vigencia','fecha_fin_vigencia']].head(5)

,fecha_emision,fecha_inicio_vigencia,fecha_fin_vigencia
0,2021-11-23,2021-11-23,2022-11-23
1,2019-08-19,2019-08-19,2020-08-19
2,2022-07-11,2022-07-11,2023-07-11
3,2019-03-12,2019-03-12,2020-03-12
4,2020-11-03,2020-11-03,2021-11-03


In [7]:
# Revisamos
print(f"Num duplicados: {df2_copy.duplicated().sum()}")
print(f"Sexo diferente: {df2_copy['sexo'].isin(['MASCULINO','FEMENINO','m','f']).sum()}")
print(f"Fechas como str: {df2_copy.select_dtypes('object').filter(like='fecha').shape[1]}")

Num duplicados: 0
Sexo diferente: 0
Fechas como str: 0


In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# FASE 3: ENRIQUECIMIENTO
# ══════════════════════════════════════════════════════════════════════════════
# 3a. Merge con catalogo_ramos: agregar nombre_largo, tasa_base
# 3b. Merge con catalogo_agentes: agregar nombre del agente, region
# 3c. Crear: g_edad con pd.cut
# 3d. Crear: prima_calc = suma_asegurada * tasa_base * 1.16
# 3e. Crear: nivel_riesgo con .apply(clasificar_riesgo) — de mi_modulo
# 3f. Crear: edad_calc desde fecha_nacimiento
# 3g. Crear: dias_vigencia, fraccion_expuesta

# Tu codigo aqui:
# calculamos el numero de siniestros
sin = siniestros.groupby('id_poliza').agg(
    siniestros = ('id_poliza','count')
).reset_index()
sin.head(5)

,id_poliza,siniestros
0,POL-000005,1
1,POL-000011,1
2,POL-000018,2
3,POL-000020,1
4,POL-000023,1


In [ ]:
# 3a. Merge con catalogo_ramos: agregar nombre_largo, tasa_base
cart_ramos = pd.merge(df2_copy, cat_ramos[['ramo','nombre_largo','tasa_base']], on='ramo', how='left')

# 3b. Merge con catalogo_agentes: agregar nombre del agente, region
cart_agente = pd.merge(cart_ramos, cat_agentes[['agente_id', 'nombre','region']], on='agente_id', how='left')

cart_total = pd.merge(cart_agente, sin, on='id_poliza', how='left')
cart_total['siniestros'] = cart_total['siniestros'].fillna(0)

# 3c. Crear: g_edad con pd.cut
cart_total['g_edad'] = pd.cut(cart_total['edad'],bins=[0,30,45,60,100],labels=['18-30','31-45','46-60','61+'])

# 3d. Crear: prima_calc = suma_asegurada * tasa_base * 1.16
cart_total['prima_calc'] = cart_total['suma_asegurada'] * cart_total['tasa_base'] * 1.16

# 3e. Crear: nivel_riesgo con .apply(clasificar_riesgo) — de mi_modulo
import mi_modulo
from mi_modulo import clasificar_riesgo
cart_total['nivel_riesgo'] = cart_total['siniestros'].apply(clasificar_riesgo)

# 3f. Crear: edad_calc desde fecha_nacimiento
fecha_hoy = pd.Timestamp.today()

cart_total['edad_calc'] = ((fecha_hoy - cart_total['fecha_nacimiento']).dt.days / 365.25)

# 3g. Crear: dias_vigencia, fraccion_expuesta
cart_total['dias_vigencia'] = (cart_total['fecha_fin_vigencia'] - cart_total['fecha_inicio_vigencia']).dt.days

dias_transcurridos = (fecha_hoy - cart_total['fecha_inicio_vigencia']).dt.days
cart_total['fraccion_expuesta'] = (dias_transcurridos / cart_total['dias_vigencia']).clip(0, 1).round(4)

print(f"Numero de registros: {len(cart_total)}")
cart_total.head(3)

Numero de registros: 50000


,id_poliza,num_poliza,nombre_x,apellido_paterno,apellido_materno,rfc,fecha_nacimiento,edad,sexo,estado_civil,...,tasa_base,nombre_y,region,siniestros,g_edad,prima_calc,nivel_riesgo,edad_calc,dias_vigencia,fraccion_expuesta
0,POL-000001,Vid-21-000001,Gabriela,Moreno,Vega,MOGV020429CG6,2002-04-29,24,F,Union libre,...,0.018,Enrique Castillo,Chihuahua,0.0,18-30,62640.0,BAJO,24.046543,365,1.0
1,POL-000002,Aut-19-000002,Valeria,Torres,Castillo,TOVC020815IA8,2002-08-15,23,F,Casado,...,0.035,Patricia Rios,Veracruz,0.0,18-30,6090.0,BAJO,23.750856,366,1.0
2,POL-000003,GMM-22-000003,Fernanda,Ramos,Silva,RAFS941018BC1,1994-10-18,31,M,Union libre,...,0.022,Manuel Mendoza,Chihuahua,0.0,31-45,20416.0,BAJO,31.575633,365,1.0


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FASE 4: ANALISIS Y REPORTES
# ══════════════════════════════════════════════════════════════════════════════
# 4a. groupby+agg por ramo: polizas, prima_total, prima_prom, pct_cartera
# 4b. groupby+agg por agente: polizas, prima_total, comision (10%)
# 4c. pivot_table prima por ramo x g_edad con margins=True
# 4d. pivot_table polizas por estado x ramo
# 4e. Identifica: ramo con mayor prima total y zona con mayor frecuencia

# Tu codigo aqui:
# 4a. groupby+agg por ramo: polizas, prima_total, prima_prom, pct_cartera
agrup_ramo = cart_total.groupby('ramo').agg(
    polizas    =('id_poliza','count'),
    prima_total=('prima_total','sum'),
    prima_prom =('prima_total','mean'),
).round(2).reset_index()
agrup_ramo['pct_cartera'] = (agrup_ramo['prima_total']/agrup_ramo['prima_total'].sum()*100).round(1)
print(
    agrup_ramo
    .sort_values(by='pct_cartera'))

print('La mayor participacion de la cartera y mayor prima total la ocupa GMM')

                    ramo  polizas   prima_total  prima_prom  pct_cartera
0  Accidentes Personales     5207  2.498068e+07     4797.52          1.7
1                  Autos    14752  2.538235e+08    17206.04         17.3
3                   Vida     7510  4.543321e+08    60496.95         31.0
2                    GMM    22531  7.310223e+08    32445.18         49.9
La mayor participacion de la cartera y mayor prima total la ocupa GMM


In [ ]:
# 4b. groupby+agg por agente: polizas, prima_total, comision (10%)
agrup_agente = cart_total.groupby('agente_id').agg(
    polizas    =('id_poliza','count'),
    prima_total=('prima_total','sum')

).reset_index()
print('Top 5 de agentes con mas polizas asociadas:')
print(agrup_agente.sort_values(by='polizas', ascending=False).head(5))

Top 5 de agentes con mas polizas asociadas:
   agente_id  polizas  prima_total
32     AG033      684  19852287.17
11     AG012      679  20269987.60
40     AG041      674  21039960.22
75     AG076      668  20065957.85
41     AG042      665  18884582.93


In [ ]:
# 4c. pivot_table prima por ramo x g_edad con margins=True
prima_ramo = pd.pivot_table(
    cart_total,
    values   = 'prima_total',
    index    = 'ramo',
    columns  = 'g_edad',
    aggfunc  = 'sum',
    fill_value = 0,
    margins    = True,
    margins_name = 'TOTAL'
).round(0) / 1_000 
print(prima_ramo.to_string())

g_edad                      18-30       31-45       46-60         61+        TOTAL
ramo                                                                              
Accidentes Personales    5248.369    8348.395    8731.668    2652.247    24980.679
Autos                   55457.022   84225.512   84571.404   29569.569   253823.506
GMM                    139278.258  223908.688  264218.298  103617.078   731022.322
Vida                    79667.849  134774.641  166301.773   73587.860   454332.123
TOTAL                  279651.498  451257.236  523823.143  209426.754  1464158.631


/var/folders/k1/_jflqvn90v31fwxjw3lf6y680000gn/T/ipykernel_6194/2209731662.py:1: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  prima_ramo = pd.pivot_table(


In [ ]:
# 4d. pivot_table polizas por estado x ramo
poliza_estado = pd.pivot_table(
    cart_total,
    values   = 'id_poliza',
    index    = 'estado',
    columns  = 'ramo',
    aggfunc  = 'count',
    fill_value = 0,
    margins    = True,
    margins_name = 'TOTAL'
).round(0).sort_values(by='TOTAL', ascending=False)

print(poliza_estado.to_string())
print(f"El estado con mayor frecuencia de siniestros en conjunto de los 4 ramos es Guanajuato")

ramo              Accidentes Personales  Autos    GMM  Vida  TOTAL
estado                                                            
TOTAL                              5207  14752  22531  7510  50000
Guanajuato                          340   1033   1579   493   3445
Sonora                              367    999   1479   522   3367
Coahuila                            361    999   1492   514   3366
Queretaro                           351    990   1537   488   3366
CDMX                                357   1000   1485   515   3357
Jalisco                             344    959   1526   523   3352
Chihuahua                           360    972   1501   517   3350
Baja California                     321    990   1508   529   3348
Veracruz                            345   1009   1507   477   3338
Estado de Mexico                    325    977   1475   547   3324
Yucatan                             376    938   1506   502   3322
Michoacan                           344    982   1498   470   

/var/folders/k1/_jflqvn90v31fwxjw3lf6y680000gn/T/ipykernel_6194/1285784250.py:1: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  poliza_estado = pd.pivot_table(


In [21]:
# 4e. Identifica: ramo con mayor prima total y zona con mayor frecuencia
print("El ramo con mayor prima total es GMM y la zona/estado con mayor frecuencia de siniestros es Guanajuato ")

El ramo con mayor prima total es GMM y la zona/estado con mayor frecuencia de siniestros es Guanajuato 


In [59]:
pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]1/2 [openpyxl]
Note: you may need to restart the kernel to use updated packages.


In [13]:
# ══════════════════════════════════════════════════════════════════════════════
# FASE 5: EXPORTAR
# ══════════════════════════════════════════════════════════════════════════════
# Excel con 5 hojas: Cartera_Limpia, Resumen_Ramo, Resumen_Agente,
#                    Pivot_Prima, Pivot_Zona
# Parquet: cartera_q1_2026_final.parquet
# Compara tamano CSV equivalente vs Parquet

# Tu codigo aqui:

with pd.ExcelWriter('datos/reporte_q1_2026.xlsx', engine='openpyxl') as writer:
    cart_total.to_excel(writer, sheet_name='Cartera_limpia', index=False)
    agrup_ramo.to_excel(writer, sheet_name='Resumen_Ramo', index=False)
    agrup_agente.to_excel(writer, sheet_name='Resumen_Agente', index=False)
    prima_ramo.to_excel(writer, sheet_name='Pivot_Prima')
    poliza_estado.to_excel(writer, sheet_name='Pivot_Canal')

kb_excel = os.path.getsize('datos/reporte_q1_2026.xlsx')/1024
cart_total.to_parquet('datos/cartera_q1_2026_final.parquet', index=False)
kb_parquet = os.path.getsize('datos/cartera_q1_2026_final.parquet')/1024
print(f'Tamaño del excel: {kb_excel:.0f} KB con 5 hojas')
print(f'Tamaño del parquet: {kb_parquet:.0f} KB ')

Tamaño del excel: 12608 KB con 5 hojas
Tamaño del parquet: 3189 KB 


In [ ]:
# ── Solucion resumida (descomenta solo si necesitas referencia) ──────────────

# FASE 1:
# df = pd.read_csv('datos/cartera_polizas.csv', usecols=ANALITICAS, na_values=['N/D','N/A'])
# ramos_cat = pd.read_csv('datos/catalogo_ramos.csv')
# agentes_cat = pd.read_csv('datos/catalogo_agentes.csv')

# FASE 2:
# df = df.drop_duplicates()
# df['sexo'] = df['sexo'].str.strip().str.upper().map(MAPA_SEXO)
# for col in ['fecha_emision','fecha_inicio_vigencia','fecha_fin_vigencia']:
#     df[col] = pd.to_datetime(df[col], errors='coerce')
# df['fecha_nacimiento'] = pd.to_datetime(df['fecha_nacimiento'],format='%d/%m/%Y',errors='coerce')
# df['prima_neta'] = df.groupby('ramo')['prima_neta'].transform(lambda x: x.fillna(x.median()))
# df['codigo_postal'] = df['codigo_postal'].replace('N/D', np.nan)
# for col in ['ramo','plan','canal_venta','forma_pago','estado','sexo','tipo_vehiculo']:
#     if col in df.columns: df[col] = df[col].astype('category')

# FASE 3 (extracto):
# df = pd.merge(df, ramos_cat[['ramo','nombre_largo','tasa_base']], on='ramo', how='left')
# df['g_edad'] = pd.cut(df['edad'],bins=[0,30,45,60,100],labels=['18-30','31-45','46-60','61+'])
# df['prima_calc'] = df['suma_asegurada'] * df['tasa_base'] * 1.16
# from mi_modulo import clasificar_riesgo
# df['nivel_riesgo'] = df['prima_calc'].apply(lambda p: 'ALTO' if p>15000 else 'MEDIO' if p>6000 else 'BAJO')

# FASE 5:
# with pd.ExcelWriter('datos/reporte_ejecutivo_Q1_2026.xlsx', engine='openpyxl') as w:
#     df.to_excel(w,'Cartera_Limpia',index=False)
#     resumen_ramo.to_excel(w,'Resumen_Ramo',index=False)
#     resumen_agente.to_excel(w,'Resumen_Agente',index=False)
#     tabla_prima.to_excel(w,'Pivot_Prima')
#     tabla_zona.to_excel(w,'Pivot_Zona')
# df.to_parquet('datos/cartera_q1_2026_final.parquet', index=False)
# print(f'CSV: {df.to_csv(index=False).encode().__len__()/1024:.0f} KB')
# print(f'Parquet: {os.path.getsize("datos/cartera_q1_2026_final.parquet")/1024:.0f} KB')

---
## Resumen: Lo que Aprendiste en la Sesion 8

| Duda | Herramienta | Aprendizaje clave |
|------|-------------|-------------------|
| 46 columnas | `usecols` + taxonomia | Clasificar antes de cargar — decision de negocio |
| Texto sucio | `str.strip/upper/map` | Normalizar ANTES del primer groupby |
| Fechas texto | `pd.to_datetime(errors='coerce')` | Nunca detener el pipeline por fechas invalidas |
| JSON anidado | `json_normalize(sep='_')` | Aplanar antes de analizar |
| 90k filas | `chunksize` + `category` | Medir memoria antes de decidir |
| Downcast riesgoso | Regla float32 < 100k MXN | float64 para montos grandes siempre |
| Polars | `pl.scan_csv().collect()` | Lazy evaluation = optimizacion automatica |

**T5 Pandas — COMPLETADO**

**Proxima sesion — Mie 6 mayo, 18:00 h:**
T6 Visualizacion — Matplotlib, Seaborn y Plotly.

```bash
git add sesion8_M1_notebook.ipynb
git commit -m "Sesion 8: pipeline completo datos reales - str datetime JSON Polars"
git push
```

---
*Diplomado ML en Seguros · FC UNAM · 2026*